# Job Market Analytics — Exploratory Data Analysis Sandbox

This notebook serves as a sandbox for analyzing the scraped and cleaned job postings stored in our MySQL database. It demonstrates:
1. Connecting to the MySQL database.
2. Loading data into Pandas DataFrames.
3. Running exploratory statistics (top skills, average salaries, remote ratio).
4. Plotting interactive visualizations.

In [ ]:
import os
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Load environment variables
load_dotenv(dotenv_path="../.env")

# DB Connection
DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "MyNewPassword123!")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME", "job_market_db")
DB_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(DB_URL)
print("Database engine connected successfully!")

## 1. Load Data

In [ ]:
df_jobs = pd.read_sql("""
    SELECT jp.*, c.company_name, l.city, l.region, l.country 
    FROM job_postings jp 
    JOIN companies c ON jp.company_id = c.company_id 
    LEFT JOIN locations l ON jp.location_id = l.location_id
""", engine)

df_skills = pd.read_sql("""
    SELECT js.job_id, s.skill_name, s.skill_category 
    FROM job_skills js 
    JOIN skills s ON js.skill_id = s.skill_id
""", engine)

print(f"Loaded {len(df_jobs)} jobs and {len(df_skills)} skills mappings.")
df_jobs.head()

## 2. Basic Stats & Distributions

In [ ]:
# Total jobs by country
print("\n--- Job Count by Country ---")
print(df_jobs['country'].value_counts())

# Job categories
print("\n--- Job Categories ---")
print(df_jobs['job_category'].value_counts())

# Remote ratio
print("\n--- Work Mode ---")
print(df_jobs['is_remote'].value_counts(normalize=True) * 100)

## 3. Visualize Salary by Category (USD)

In [ ]:
df_salaries = df_jobs[df_jobs['salary_mid_usd'].notna()]
if not df_salaries.empty:
    fig = px.box(df_salaries, x="job_category", y="salary_mid_usd", color="job_category",
                 title="Salary Distribution by Category (USD)",
                 labels={"job_category": "Job Category", "salary_mid_usd": "Salary Midpoint (USD)"})
    fig.show()
else:
    print("No salary data found in database.")

## 4. Top Skills by Frequency

In [ ]:
tech_skills = df_skills[df_skills['skill_category'] == 'technical']
top_tech = tech_skills['skill_name'].value_counts().head(15).reset_index()
top_tech.columns = ['Skill', 'Count']

fig_skills = px.bar(top_tech, x='Count', y='Skill', orientation='h', 
                    title='Top 15 Technical Skills in Job Listings',
                    color='Count', color_continuous_scale='Blues')
fig_skills.update_layout(yaxis={'categoryorder':'total ascending'})
fig_skills.show()